In [ ]:
# ensure gpu
!nvidia-smi

Sun Feb 22 21:28:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# install packages
# !pip install -U transformers==4.57.1 torch pillow einops torchvision accelerate decord2 molmo_utils bitsandbytes

In [ ]:
# get geometry3k TEST split
!wget https://lupantech.github.io/inter-gps/geometry3k/test.zip
!unzip test.zip

--2026-02-22 21:28:39--  https://lupantech.github.io/inter-gps/geometry3k/test.zip
Resolving lupantech.github.io (lupantech.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to lupantech.github.io (lupantech.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 25524601 (24M) [application/x-zip-compressed]
Saving to: ‘test.zip’

test.zip            100%[===================>]  24.34M  --.-KB/s    in 0.05s   

2026-02-22 21:28:40 (474 MB/s) - ‘test.zip’ saved [25524601/25524601]

Archive:  test.zip
   creating: test/
   creating: test/2925/
  inflating: test/2925/logic_form.json  
  inflating: test/2925/img_diagram_point.png  
  inflating: test/2925/data.json     
  inflating: test/2925/img_diagram.png  
   creating: test/2513/
  inflating: test/2513/logic_form.json  
  inflating: test/2513/img_diagram_point.png  
  inflating: test/2513/data.json     
  inflating: test/2513/img_diagram.png  
   creating: test/24

In [ ]:
import re
import os
import json
import copy
import torch
from PIL import Image
from torch.utils.data import DataLoader
from transformers import AutoProcessor, AutoModelForCausalLM, AutoTokenizer

In [ ]:
model_id="Qwen/Qwen2.5-7B-Instruct"

# load the processor
processor = AutoProcessor.from_pretrained(
    model_id,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    device_map={"": "cuda"}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [ ]:
test_problems = []

# Get a list of all subdirectories within the 'test/' directory and filter for numerical directories
problem_ids = sorted([d for d in os.listdir('test/') if d.isdigit()], key=int)

for problem_id in problem_ids:
    problem_dir = os.path.join('test/', problem_id)
    data_json_path = os.path.join(problem_dir, 'data.json')
    img_diagram_path = os.path.join(problem_dir, 'img_diagram.png')

    if os.path.exists(data_json_path):
        with open(data_json_path, 'r') as f:
            data = json.load(f)

        problem_text = data.get('problem_text', '')
        choices = data.get('choices', [])
        answer = data.get('answer', '')

        # Create a dictionary to map choice indices to letters
        choice_letters = {0: 'A. ', 1: 'B. ', 2: 'C. ', 3: 'D. ', 4: 'E. ', 5: 'F. '}
        formatted_choices = [
            f"{choice_letters.get(i, str(i) + '.')}" + c for i, c in enumerate(choices)
        ]

        test_problems.append({
            'problem_id': problem_id,
            'problem_text': problem_text,
            'choices': formatted_choices,
            'answer': answer,
            'logic_form_path': os.path.join(problem_dir, 'logic_form.json')
        })

print(f"Loaded {len(test_problems)} test problems.")
# Display the first problem to verify structure
if test_problems:
    print("\nFirst processed problem example:")
    print(json.dumps(test_problems[0], indent=2))

Loaded 601 test problems.

First processed problem example:
{
  "problem_id": "2401",
  "problem_text": "Find the area of the figure.",
  "choices": [
    "A. 30",
    "B. 60",
    "C. 120",
    "D. 240"
  ],
  "answer": "B",
  "logic_form_path": "test/2401/logic_form.json"
}


In [ ]:
def translate_predicate(pred):
    # 1. Equals(LengthOf(Line(A, B)), 13) or (x+5)
    length_match = re.search(r"Equals\(LengthOf\(Line\((\w),\s*(\w)\)\),\s*([^)]+)\)", pred)
    if length_match:
        p1, p2, val = length_match.groups()
        return f"Length of {p1}{p2} is {val.strip()}."

    # 2. Equals(MeasureOf(Angle(A, B, C)), 90) or (x-1)
    angle_match = re.search(r"Equals\(MeasureOf\(Angle\((\w),\s*(\w),\s*(\w)\)\),\s*([^)]+)\)", pred)
    if angle_match:
        p1, p2, p3, val = angle_match.groups()
        return f"Angle {p1}{p2}{p3} is {val.strip()} degrees."

    # 3. Parallel(Line(B, C), Line(E, D))
    parallel_match = re.search(r"Parallel\(Line\((\w),\s*(\w)\),\s*Line\((\w),\s*(\w)\)\)", pred)
    if parallel_match:
        a, b, c, d = parallel_match.groups()
        return f"Line {a}{b} is parallel to line {c}{d}."

    # 4. PointLiesOnCircle(T, Circle(M, radius_5_0))
    circle_match = re.search(r"PointLiesOnCircle\((\w),\s*Circle\((\w),\s*([^)]+)\)\)", pred)
    if circle_match:
        pt, center, radius = circle_match.groups()
        # Optional: Clean up "radius_5_0" to "radius 5.0"
        clean_radius = radius.replace('_', ' ')
        return f"Point {pt} lies on circle {center} with {clean_radius}."

    # 5. PointLiesOnLine(D, Line(A, C))
    lies_on_match = re.search(r"PointLiesOnLine\((\w),\s*Line\((\w),\s*(\w)\)\)", pred)
    if lies_on_match:
        pt, l1, l2 = lies_on_match.groups()
        return f"Point {pt} lies on line {l1}{l2}."

    # 6. Perpendicular(Line(B, D), Line(D, C))
    perp_match = re.search(r"Perpendicular\(Line\((\w),\s*(\w)\),\s*Line\((\w),\s*(\w)\)\)", pred)
    if perp_match:
        a, b, c, d = perp_match.groups()
        return f"Line {a}{b} is perpendicular to {c}{d}."

    return pred

def linearize_logic(logic_path):
    """Parses logic_form.json with a predicate-to-text mapping."""
    try:
        with open(logic_path, 'r') as f:
            data = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return ""

    descriptions = []

    if "diagram_logic_form" in data:
        for item in data["diagram_logic_form"]:
            descriptions.append(translate_predicate(item))

    full_string = " ".join(descriptions)

    return full_string

In [ ]:
prompt_direct = """Your output response MUST ONLY be a single letter from the choices: A, B, C, or D.
    For example: \nStep 1: reasoning step 1. \nStep 2: reasoning step 2. \nStep 3: reasoning step 3.\n...\nFinal Answer: <answer>...</answer>
    \n\nQuestion: {problem_text}\n\nChoices: {choices_text}\n\n
    Answer:"""
prompt_thinking = """You are a helpful AI assistant.
    When answering questions, you must always show your step-by-step reasoning process first.
    Your final answer MUST by a single letter from the choices (e.g., A, B, C, or D) in the following format: <answer>n</answer> where n is the letter choice (e.g., A, B, C, or D) corresponding to the selected answer.
    \n\nQuestion: {problem_text}\n\nChoices: {choices_text}\n\n
    """
prompt_direct_with_logic = """Your output response MUST ONLY be a single letter from the choices: A, B, C, or D.
    For example: \nStep 1: reasoning step 1. \nStep 2: reasoning step 2. \nStep 3: reasoning step 3.\n...\nFinal Answer: <answer>...</answer>
    \n\nQuestion: {problem_text}\n\nLogic: {logic}\n\nChoices: {choices_text}\n\n
    Answer:"""
prompt_thinking_with_logic = """You are a helpful AI assistant.
    When answering questions, you must always show your step-by-step reasoning process first.
    Your final answer MUST by a single letter from the choices (e.g., A, B, C, or D) in the following format: <answer>n</answer> where n is the letter choice (e.g., A, B, C, or D) corresponding to the selected answer.
    \n\nQuestion: {problem_text}\n\nLogic: {logic}\n\nChoices: {choices_text}\n\n
    """

In [ ]:
def format_problem(prompt_format, test_problems):
    test_problems_copy = copy.deepcopy(test_problems)
    for problem in test_problems_copy:
        problem_text = problem['problem_text']
        choices_text = '\n'.join(problem['choices'])
        logic = linearize_logic(problem['logic_form_path'])

        prompt = prompt_format.format(problem_text=problem_text, logic=logic, choices_text=choices_text)

        problem['model_prompt'] = [dict(type="text", text=prompt)]

    # Display the prompt for the first problem to verify structure
    if test_problems_copy:
        print("\nFirst problem's constructed prompt example:")
        print(test_problems_copy[0]['model_prompt'])
    return test_problems_copy

In [ ]:
test_problems_direct = format_problem(prompt_direct, test_problems)

test_problems_thinking = format_problem(prompt_thinking, test_problems)

test_problems_direct_with_logic = format_problem(prompt_direct_with_logic, test_problems)

test_problems_thinking_with_logic = format_problem(prompt_thinking_with_logic, test_problems)


First problem's constructed prompt example:
[{'type': 'text', 'text': 'Your output response MUST ONLY be a single letter from the choices: A, B, C, or D.\n    For example: \nStep 1: reasoning step 1. \nStep 2: reasoning step 2. \nStep 3: reasoning step 3.\n...\nFinal Answer: <answer>...</answer>\n    \n\nQuestion: Find the area of the figure.\n\nChoices: A. 30\nB. 60\nC. 120\nD. 240\n\n\n    Answer:'}]

First problem's constructed prompt example:
[{'type': 'text', 'text': 'You are a helpful AI assistant.\n    When answering questions, you must always show your step-by-step reasoning process first.\n    Your final answer MUST by a single letter from the choices (e.g., A, B, C, or D) in the following format: <answer>n</answer> where n is the letter choice (e.g., A, B, C, or D) corresponding to the selected answer.\n    \n\nQuestion: Find the area of the figure.\n\nChoices: A. 30\nB. 60\nC. 120\nD. 240\n\n\n    '}]

First problem's constructed prompt example:
[{'type': 'text', 'text': 'Y

In [ ]:
import torch
import json
import os # Added for path handling
from torch.utils.data import DataLoader

# --- Setup ---
BATCH_SIZE = 1
MAX_NEW_TOKENS = 1
SAVE_INTERVAL = 100  # Save every 100 examples
OUTPUT_FILE = "predictions_direct.json"

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def identity_collate(batch):
    return {key: [d[key] for d in batch] for key in batch[0]}

data_loader = DataLoader(
    test_problems_direct,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=identity_collate
)

predictions_direct = []
processed_count = 0  # Counter to track progress

# --- Generation Loop ---
for batch in data_loader:
    messages_batch = [
        [
            {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
            {"role": "user", "content": p[0]['text'] if isinstance(p, list) else p}
        ] for p in batch['model_prompt']
    ]

    texts = [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        for msg in messages_batch
    ]

    model_inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id
        )

    input_len = model_inputs.input_ids.shape[1]
    response_ids = generated_ids[:, input_len:]
    decoded_outputs = tokenizer.batch_decode(response_ids, skip_special_tokens=True)

    for i, full_text in enumerate(decoded_outputs):
        problem_id = batch['problem_id'][i]
        clean_prediction = full_text.strip()

        predictions_direct.append({
            'problem_id': problem_id,
            'prediction': clean_prediction
        })

        processed_count += 1

    # --- Periodic Saving ---
    # We check if we've crossed a 100-item threshold within this batch
    if processed_count >= SAVE_INTERVAL and (processed_count // SAVE_INTERVAL) > ((processed_count - BATCH_SIZE) // SAVE_INTERVAL):
        with open(OUTPUT_FILE, "w") as f:
            json.dump(predictions_direct, f, indent=4)
        print(f"--- Checkpoint saved at {processed_count} examples ---")
    torch.cuda.empty_cache()

# --- Final Save ---
with open(OUTPUT_FILE, "w") as f:
    json.dump(predictions_direct, f, indent=4)
print(f"Final results saved. Total: {len(predictions_direct)}")

--- Checkpoint saved at 100 examples ---
--- Checkpoint saved at 200 examples ---
--- Checkpoint saved at 300 examples ---
--- Checkpoint saved at 400 examples ---
--- Checkpoint saved at 500 examples ---
--- Checkpoint saved at 600 examples ---
Final results saved. Total: 601


In [ ]:
import torch
import json
import os # Added for path handling
from torch.utils.data import DataLoader

# --- Setup ---
BATCH_SIZE = 4
MAX_NEW_TOKENS = 1
SAVE_INTERVAL = 100  # Save every 100 examples
OUTPUT_FILE = "predictions_direct_with_logic.json"

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def identity_collate(batch):
    return {key: [d[key] for d in batch] for key in batch[0]}

data_loader = DataLoader(
    test_problems_direct_with_logic,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=identity_collate
)

predictions_direct = []
processed_count = 0  # Counter to track progress

# --- Generation Loop ---
for batch in data_loader:

    messages_batch = [
        [
            {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
            {"role": "user", "content": p[0]['text'] if isinstance(p, list) else p}
        ] for p in batch['model_prompt']
    ]

    texts = [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        for msg in messages_batch
    ]

    model_inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id
        )

    input_len = model_inputs.input_ids.shape[1]
    response_ids = generated_ids[:, input_len:]
    decoded_outputs = tokenizer.batch_decode(response_ids, skip_special_tokens=True)

    for i, full_text in enumerate(decoded_outputs):
        problem_id = batch['problem_id'][i]
        clean_prediction = full_text.strip()

        predictions_direct.append({
            'problem_id': problem_id,
            'prediction': clean_prediction
        })

        processed_count += 1

    # --- Periodic Saving ---
    # We check if we've crossed a 100-item threshold within this batch
    if processed_count >= SAVE_INTERVAL and (processed_count // SAVE_INTERVAL) > ((processed_count - BATCH_SIZE) // SAVE_INTERVAL):
        with open(OUTPUT_FILE, "w") as f:
            json.dump(predictions_direct, f, indent=4)
        print(f"--- Checkpoint saved at {processed_count} examples ---")

# --- Final Save ---
with open(OUTPUT_FILE, "w") as f:
    json.dump(predictions_direct, f, indent=4)
print(f"Final results saved. Total: {len(predictions_direct)}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


--- Checkpoint saved at 100 examples ---
--- Checkpoint saved at 200 examples ---
--- Checkpoint saved at 300 examples ---
--- Checkpoint saved at 400 examples ---
--- Checkpoint saved at 500 examples ---
--- Checkpoint saved at 600 examples ---
--- Checkpoint saved at 601 examples ---
Final results saved. Total: 601


In [ ]:
import torch
import json
import os # Added for path handling
from torch.utils.data import DataLoader

# --- Setup ---
BATCH_SIZE = 4
MAX_NEW_TOKENS = 1
SAVE_INTERVAL = 100  # Save every 100 examples
OUTPUT_FILE = "predictions_thinking_with_logic.json"

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def identity_collate(batch):
    return {key: [d[key] for d in batch] for key in batch[0]}

data_loader = DataLoader(
    test_problems_direct_with_logic,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=identity_collate
)

predictions_direct = []
processed_count = 0  # Counter to track progress

# --- Generation Loop ---
for batch in data_loader:

    messages_batch = [
        [
            {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
            {"role": "user", "content": p[0]['text'] if isinstance(p, list) else p}
        ] for p in batch['model_prompt']
    ]

    texts = [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        for msg in messages_batch
    ]

    model_inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id
        )

    input_len = model_inputs.input_ids.shape[1]
    response_ids = generated_ids[:, input_len:]
    decoded_outputs = tokenizer.batch_decode(response_ids, skip_special_tokens=True)

    for i, full_text in enumerate(decoded_outputs):
        problem_id = batch['problem_id'][i]
        clean_prediction = full_text.strip()

        predictions_direct.append({
            'problem_id': problem_id,
            'prediction': clean_prediction
        })

        processed_count += 1

    # --- Periodic Saving ---
    # We check if we've crossed a 100-item threshold within this batch
    if processed_count >= SAVE_INTERVAL and (processed_count // SAVE_INTERVAL) > ((processed_count - BATCH_SIZE) // SAVE_INTERVAL):
        with open(OUTPUT_FILE, "w") as f:
            json.dump(predictions_direct, f, indent=4)
        print(f"--- Checkpoint saved at {processed_count} examples ---")

# --- Final Save ---
with open(OUTPUT_FILE, "w") as f:
    json.dump(predictions_direct, f, indent=4)
print(f"Final results saved. Total: {len(predictions_direct)}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'min_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Processed ID: 2401 | Extracted: B
Processed ID: 2402 | Extracted: A
Processed ID: 2403 | Extracted: None
Processed ID: 2404 | Extracted: None
Processed ID: 2405 | Extracted: None
Processed ID: 2406 | Extracted: None
Processed ID: 2407 | Extracted: C
Processed ID: 2408 | Extracted: None
Processed ID: 2409 | Extracted: C
Processed ID: 2410 | Extracted: None
Processed ID: 2411 | Extracted: B
Processed ID: 2412 | Extracted: None
Processed ID: 2413 | Extracted: B
Processed ID: 2414 | Extracted: B
Processed ID: 2415 | Extracted: None
Processed ID: 2416 | Extracted: B
Processed ID: 2417 | Extracted: C
Processed ID: 2418 | Extracted: D
Processed ID: 2419 | Extracted: B
Processed ID: 2420 | Extracted: D
Processed ID: 2421 | Extracted: C
Processed ID: 2422 | Extracted: None
Processed ID: 2423 | Extracted: C
Processed ID: 2424 | Extracted: None
Processed ID: 2425 | Extracted: D
Processed ID: 2426 | Extracted: C
Processed ID: 2427 | Extracted: A
Processed ID: 2428 | Extracted: B
Processed ID: 2429